# Visual encoding
Portable panels based on the manuscript's encoding analysis (`draw_pic/fig2.ipynb`).
Compute predictions from a **release checkpoint** or load archived predictions explicitly.
Panels show measured/predicted waveforms, spectra, sensor correlations, phase-lag indices,
and optional EEG band-power maps. Blue: measured; red dashed: predicted.
Amplitudes remain in preprocessed units. Matrix similarity is descriptive, not connectivity validation.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run from this notebook directory or the release root.
ROOT = Path.cwd() if (Path.cwd() / "brainae").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "notebooks"))
from figure_utils import *
style()
OUTPUT = ROOT / "figures"


In [ ]:
MODALITY = 'eeg'
DATA = ROOT / 'data/prepared/eeg/sub-01'
CHECKPOINT = ROOT / 'outputs/eeg/sub-01/best.pt'
FEATURES = ROOT / 'data/features/eeg_test.npy'
DEVICE = 'cpu'
SFREQ = {'eeg': 250, 'meg': 200, 'mua': 1000}[MODALITY]
T0 = .005 if MODALITY == 'meg' else 0.
# Replace with verified indices from your channel metadata; no regions are guessed.
REGIONS = {}  # e.g. {'Occipital': [indices], 'Temporal': [indices], 'Parietal': [indices]}
CHANNEL_NAMES = DATA / 'channel_names.npy'
MONTAGE = 'standard_1005'


In [ ]:
from brainae.runner import feature_array
measured = responses(DATA)
ids = np.load(DATA / 'test_ids.npy', allow_pickle=False)
features = feature_array(FEATURES, ids, assume_order=True)  # NPY: verify original row order first.
encoder, decoder = load_model(CHECKPOINT, MODALITY, DEVICE)
predicted = infer(decoder, features, DEVICE)[:, 0]
# Archived alternative: load [images, channels, samples] arrays from your own files.
assert measured.shape == predicted.shape
times = T0 + np.arange(measured.shape[-1]) / SFREQ
correlations = pearson_rows(measured, predicted)
pd.DataFrame({'image_id': ids, 'encoding_r': correlations,
              'mse': ((measured-predicted)**2).mean((1,2))}).to_csv(
    (OUTPUT.mkdir(parents=True, exist_ok=True) or OUTPUT) / 'encoding_per_image.csv', index=False)
print('Mean image-wise Pearson r:', np.nanmean(correlations))


In [ ]:
from scipy.signal import welch, hilbert
regions = REGIONS or {'All channels': list(range(measured.shape[1]))}
fig, axes = plt.subplots(len(regions), 1, figsize=(5.2, 2.3*len(regions)), squeeze=False, layout='constrained')
for i, (name, indices) in enumerate(regions.items()):
    ax = axes[i, 0]
    for values, label, color, line in [(measured, 'Measured', BLUE, '-'), (predicted, 'Predicted', RED, '--')]:
        ax.plot(times, values[:, indices].mean((0,1)), color=color, ls=line, label=label)
    ax.set(xlabel='Time (s)', ylabel='Amplitude (preprocessed)', title=name)
    panel(ax, chr(97+i))
axes[0,0].legend(ncol=2)
save(fig, OUTPUT, 'encoding_waveforms')

spectra = [welch(x, fs=SFREQ, nperseg=min(x.shape[-1], int(SFREQ)), axis=-1) for x in [measured, predicted]]
fig, ax = plt.subplots(figsize=(4.2,2.7), layout='constrained')
for (f, p), label, color, line in zip(spectra, ['Measured', 'Predicted'], [BLUE, RED], ['-', '--']):
    ax.semilogy(f, np.maximum(p.mean((0,1)), 1e-15), color=color, ls=line, label=label)
ax.set(xlim=(0,min(95,SFREQ/2)), xlabel='Frequency (Hz)', ylabel='PSD (preprocessed units²/Hz)')
ax.legend(); save(fig, OUTPUT, 'encoding_spectrum')

# Trial-averaged response matrices, matching the original figure's averaging.
def pli(x):
    phase = np.angle(hilbert(x, axis=-1))
    return np.stack([np.abs(np.mean(np.sign(np.sin(p[None,:]-phase)), axis=-1)) for p in phase])
fig, axes = plt.subplots(2,2,figsize=(6.2,5.4),layout='constrained')
for row, (x, label) in enumerate([(measured,'Measured'),(predicted,'Predicted')]):
    mean = x.mean(0)
    for col, (matrix,title,cmap,lo,hi) in enumerate([(np.corrcoef(mean),'Sensor correlation','RdBu_r',-1,1), (pli(mean),'Phase-lag index','viridis',0,1)]):
        ax=axes[row,col]; im=ax.imshow(matrix,cmap=cmap,vmin=lo,vmax=hi, interpolation='nearest')
        ax.set(title=f'{label}: {title}', xlabel='Channel', ylabel='Channel')
        panel(ax,chr(97+row*2+col)); fig.colorbar(im,ax=ax,shrink=.7)
save(fig, OUTPUT, 'encoding_sensor_structure')


## Optional EEG band-power maps
Requires real channel names in exactly the array order and MNE preprocessing dependencies.
Color limits are shared between measured and predicted responses within each band.
No sensor positions are invented. Maps describe the preprocessed coordinates.

In [ ]:
if MODALITY == 'eeg' and CHANNEL_NAMES.exists():
    import mne
    names = np.load(CHANNEL_NAMES, allow_pickle=False).tolist()
    assert len(names) == measured.shape[1]
    info = mne.create_info(names, SFREQ, ch_types='eeg')
    info.set_montage(MONTAGE, on_missing='raise')
    bands = {'Delta':(1,4), 'Theta':(4,8), 'Alpha':(8,13), 'Beta':(13,30), 'Gamma':(30,80)}
    fig, axes = plt.subplots(2,len(bands),figsize=(10,4.1),layout='constrained')
    for col,(name,(lo,hi)) in enumerate(bands.items()):
        powers=[]
        for f,p in spectra:
            keep=(f>=lo)&(f<hi)
            powers.append(10*np.log10(np.maximum(p[:,:,keep].mean((0,2)),1e-15)))
        limits=(min(p.min() for p in powers), max(p.max() for p in powers))
        for row,p in enumerate(powers):
            im,_=mne.viz.plot_topomap(p,info,axes=axes[row,col],show=False,cmap='viridis',vlim=limits)
            axes[row,col].set_title(f'{["Measured","Predicted"][row]}: {name}', fontsize=8)
        fig.colorbar(im,ax=axes[:,col],shrink=.6,label='PSD (dB)')
    save(fig,OUTPUT,'encoding_band_topographies')
else:
    print('Topographies skipped: supply verified EEG channel names.')
